# RNNs Assignment

<a target="_blank" href="https://colab.research.google.com/github/sham-nlp/2026nlp-4-rnns/blob/main/04_rnn_assignment_student.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Name:** `Your Name Here`


**Date:** `Insert Date`

---

## Instructions

Complete the code sections marked with `# YOUR CODE HERE`.

**Submission:** Submit this notebook with all cells executed and outputs visible.

---

## Part 1: Understanding RNN Shapes

<details>
<summary>Hint 1</summary>

Use `nn.RNN(input_size, hidden_size, num_layers, batch_first=True)`

Docs: https://pytorch.org/docs/stable/generated/torch.nn.RNN.html
</details>

In [ ]:
import torch
import torch.nn as nn

# Set seed for reproducibility
torch.manual_seed(42)

# TODO: Create an RNN layer with:
# - input_size = 32
# - hidden_size = 64
# - num_layers = 2
# - batch_first = True
rnn = # YOUR CODE HERE

print(f"RNN: {rnn}")

<details>
<summary>Hint 1</summary>

Use `torch.randn(shape)` to create random tensor.

Docs: https://pytorch.org/docs/stable/generated/torch.randn.html
</details>

In [ ]:
# TODO: Create input tensor with shape (batch_size=8, seq_len=20, input_size=32)
x = # YOUR CODE HERE

print(f"Input shape: {x.shape}")  # Expected: torch.Size([8, 20, 32])

<details>
<summary>Hint 1</summary>

Call the RNN: `output, h_n = rnn(x)` (RNN returns output and hidden state only, no cell state)
</details>

In [ ]:
# TODO: Pass input through RNN
output, h_n = # YOUR CODE HERE

print(f"Output shape: {output.shape}")  # Expected: torch.Size([8, 20, 64])
print(f"Hidden state shape: {h_n.shape}")  # Expected: torch.Size([2, 8, 64])

---

## Part 2: Building an RNN Cell from Scratch

<details>
<summary>Hint 1</summary>

Use `nn.Parameter(torch.randn(output_size, input_size) * 0.01)` for weights.

Docs: https://pytorch.org/docs/stable/generated/torch.randn.html
</details>

<details>
<summary>Hint 2</summary>

- `self.W_xh = nn.Parameter(torch.randn(hidden_size, input_size) * 0.01)`
- `self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)`
- `self.b = nn.Parameter(torch.zeros(hidden_size))`

Docs: https://pytorch.org/docs/stable/generated/torch.zeros.html
</details>

<details>
<summary>Hint 3</summary>

For forward: `h_t = torch.tanh(x_t @ self.W_xh.T + h_prev @ self.W_hh.T + self.b)`
</details>

In [ ]:
class RNNCell(nn.Module):
    """A simple RNN cell: h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b)"""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        # TODO: Initialize parameters
        # Use small random values for weights (* 0.01) and zeros for bias
        self.W_xh = # YOUR CODE HERE
        self.W_hh = # YOUR CODE HERE
        self.b = # YOUR CODE HERE

    def forward(self, x_t, h_prev):
        """
        Args:
            x_t: (batch_size, input_size)
            h_prev: (batch_size, hidden_size)
        Returns:
            h_t: (batch_size, hidden_size)
        """
        # TODO: Implement the RNN equation
        h_t = # YOUR CODE HERE
        return h_t

# Test
cell = RNNCell(10, 20)
x = torch.randn(4, 10)
h = torch.zeros(4, 20)
h_new = cell(x, h)
print(f"Output shape: {h_new.shape}")  # Expected: torch.Size([4, 20])

---

## Part 3: Unrolling the RNN

<details>
<summary>Hint 1</summary>

Inside the loop: extract `x[:, t, :]`, call `self.cell()`, append output to list.
</details>

<details>
<summary>Hint 2</summary>

```python
x_t = x[:, t, :]
h = self.cell(x_t, h)
outputs.append(self.fc(h))
```
</details>

In [ ]:
class RNN(nn.Module):
    """RNN that processes a sequence using our RNNCell."""

    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = RNNCell(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_len, input_size)
        Returns:
            outputs: (batch_size, seq_len, output_size)
            h_n: (batch_size, hidden_size)
        """
        batch_size, seq_len, _ = x.shape

        # Initialize hidden state
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        outputs = []

        # TODO: Loop through time steps and apply the cell
        for t in range(seq_len):
            # YOUR CODE HERE
            pass

        outputs = torch.stack(outputs, dim=1)
        return outputs, h

# Test
rnn = RNN(input_size=10, hidden_size=20, output_size=5)
x = torch.randn(4, 15, 10)
outputs, h_n = rnn(x)
print(f"Output shape: {outputs.shape}")  # Expected: torch.Size([4, 15, 5])
print(f"Final hidden shape: {h_n.shape}")  # Expected: torch.Size([4, 20])

---

## Part 4: Character-Level Language Model

In [ ]:
# Training data
text = "the quick brown fox jumps over the lazy dog. " * 10

# Create vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

print(f"Text length: {len(text)}")
print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

<details>
<summary>Hint 1</summary>

- Embedding: `nn.Embedding(num_embeddings, embedding_dim)`
- LSTM: `nn.LSTM(embedding_dim, hidden_size, batch_first=True)`
- FC: `nn.Linear(hidden_size, vocab_size)`

Docs: https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html
</details>

<details>
<summary>Hint 2</summary>

Forward pass:
```python
x = self.embedding(x)
output, hidden = self.lstm(x, hidden)
logits = self.fc(output)
return logits, hidden
```

Docs: https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
</details>

In [ ]:
class CharLM(nn.Module):
    """Character-level language model."""

    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size

        # TODO: Define layers
        self.embedding = # YOUR CODE HERE
        self.lstm = # YOUR CODE HERE (use batch_first=True)
        self.fc = # YOUR CODE HERE

    def forward(self, x, hidden=None):
        """
        Args:
            x: (batch, seq_len) - token indices
            hidden: (h, c) tuple or None
        Returns:
            logits: (batch, seq_len, vocab_size)
            hidden: (h, c) tuple
        """
        # TODO: Implement forward pass
        # YOUR CODE HERE
        pass

    def init_hidden(self, batch_size, device):
        return (
            torch.zeros(1, batch_size, self.hidden_size, device=device),
            torch.zeros(1, batch_size, self.hidden_size, device=device)
        )

model = CharLM(vocab_size, embed_dim=32, hidden_size=64)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Prepare training sequences
seq_len = 20
inputs, targets = [], []

for i in range(0, len(text) - seq_len - 1, seq_len // 2):
    inputs.append([char_to_idx[c] for c in text[i:i+seq_len]])
    targets.append([char_to_idx[c] for c in text[i+1:i+seq_len+1]])

X = torch.tensor(inputs)
Y = torch.tensor(targets)

print(f"Training samples: {len(X)}")
print(f"Input shape: {X.shape}")

---

## Part 5: Training with Gradient Clipping

<details>
<summary>Hint 1</summary>

- Zero gradients: `optimizer.zero_grad()`
- Forward: `logits, _ = model(X)`

Docs: https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
</details>

<details>
<summary>Hint 2</summary>

Loss computation:
```python
loss = criterion(logits.view(-1, vocab_size), Y.view(-1))
```
</details>

<details>
<summary>Hint 3</summary>

- Backward: `loss.backward()`
- Clip: `nn.utils.clip_grad_norm_(model.parameters(), 1.0)`
- Step: `optimizer.step()`

Docs: https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html
</details>

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

model.train()
print(f"{'Epoch':>6} | {'Loss':>8} | {'Perplexity':>10}")
print("-" * 32)

for epoch in range(500):
    # TODO: Complete the training loop

    # Step 1: Zero gradients
    # YOUR CODE HERE

    # Step 2: Forward pass
    logits, _ = # YOUR CODE HERE

    # Step 3: Compute loss
    # Reshape logits to (batch * seq, vocab) and targets to (batch * seq,)
    loss = # YOUR CODE HERE

    # Step 4: Backward pass
    # YOUR CODE HERE

    # Step 5: Gradient clipping (max_norm=1.0)
    # YOUR CODE HERE

    # Step 6: Optimizer step
    # YOUR CODE HERE

    if (epoch + 1) % 10 == 0:
        ppl = torch.exp(loss).item()
        print(f"{epoch+1:6d} | {loss.item():8.4f} | {ppl:10.2f}")

---

## Part 6: Text Generation

In [ ]:
def generate(model, start_char, length=50, temperature=1.0):
    """Generate text from the model."""
    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        x = torch.tensor([[char_to_idx[start_char]]], device=device)
        hidden = model.init_hidden(1, device)
        result = start_char

        for _ in range(length):
            logits, hidden = model(x, hidden)
            logits = logits[0, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, 1)
            x = next_idx.unsqueeze(0)
            result += idx_to_char[next_idx.item()]

    return result

# Generate text
print("Generated text:\n")
for start in ['t', 'f', 'q']:
    generated = generate(model, start, length=60)
    print(f"Start '{start}': {generated}")
    print()

---

## Part 8: Challenge (Optional)

Implement a **vanilla RNN** (using `nn.RNN`) for sentiment classification.

The model should:
1. Embed the input tokens
2. Process with a vanilla RNN layer
3. Use the final hidden state for classification
4. Output a single logit (positive/negative sentiment)

<details>
<summary>Hint 1</summary>

Use `nn.RNN(embed_dim, hidden_size, batch_first=True)` for the vanilla RNN layer.

Docs: https://pytorch.org/docs/stable/generated/torch.nn.RNN.html
</details>

<details>
<summary>Hint 2</summary>

Structure:
```python
self.embedding = nn.Embedding(vocab_size, embed_dim)
self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True)
self.fc = nn.Linear(hidden_size, 1)
```

Docs: https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
</details>

<details>
<summary>Hint 3</summary>

Forward pass:
```python
embedded = self.embedding(x)          # (batch, seq, embed)
output, h_n = self.rnn(embedded)      # h_n: (1, batch, hidden)
logits = self.fc(h_n.squeeze(0))      # (batch, 1)
```
</details>

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        # TODO: Implement the classifier using nn.RNN
        # YOUR CODE HERE
        pass

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len) token indices
        Returns:
            logits: (batch, 1) sentiment score
        """
        # TODO: Implement forward pass
        # YOUR CODE HERE
        pass

# Test
classifier = SentimentClassifier(vocab_size=1000, embed_dim=32, hidden_size=64)
x = torch.randint(0, 1000, (8, 20))  # batch=8, seq_len=20
output = classifier(x)
print(f"Output shape: {output.shape}")  # Expected: torch.Size([8, 1])

## Training The Sentiment Analysis Model

In [ ]:
#@title Data Preperation
texts = [
    "good",  "love",  "happy",  "good movie",  "love it",
    "bad",   "hate",  "sad",    "bad movie",   "hate it",
]
labels = [1, 1, 1, 1, 1,
          0, 0, 0, 0, 0]

chars = sorted(set("".join(texts)))
char_to_idx = {c: i + 1 for i, c in enumerate(chars)}
vocab_size = len(char_to_idx) + 1

print(f"Vocab: {char_to_idx}")
print(f"Vocab size: {vocab_size}")

max_len = max(len(t) for t in texts)

def encode(text):
    indices = [char_to_idx[c] for c in text]
    return indices + [0] * (max_len - len(indices))  # pad with 0

X = torch.tensor([encode(t) for t in texts])
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Example: '{texts[0]}' → {X[0].tolist()} → label {labels[0]}")

In [ ]:
model = SentimentClassifier(vocab_size=vocab_size, embed_dim=8, hidden_size=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(200):
    logits = model(X)
    loss = loss_fn(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        preds = (torch.sigmoid(logits) > 0.5).float()
        acc = (preds == y).float().mean()
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Acc: {acc.item():.0%}")

print("\n--- Predictions ---")
test_texts = ["good", "bad", "happy", "hate", "love movie", "sad movie"]

for t in test_texts:
    encoded = encode(t)
    inp = torch.tensor([encoded])
    with torch.no_grad():
        prob = torch.sigmoid(model(inp)).item()
    sentiment = "positive" if prob > 0.5 else "negative"
    print(f"  '{t:12s}' → {prob:.3f} → {sentiment}")